# Notebook 01 — Data Collection & Exploration

Fetches historical TLE archives from **Space-Track**, propagates each TLE with SGP4 at its own epoch to produce a dense position time series, and runs exploratory data analysis.

**Output:** `data/collected/positions_<NORAD>.csv` — one file per satellite with columns:
`timestamp_utc, lat_deg, lon_deg, alt_km, vx_km_s, vy_km_s, vz_km_s, tle_age_days`

**Works on:** local machine, Google Colab (no GPU needed)

---

### What this notebook does
1. Detect Colab vs local runtime and install dependencies
2. Load Space-Track credentials (file picker or Colab Secrets)
3. Fetch up to 180 days of historical TLEs per satellite
4. Propagate each TLE at 1-minute steps over its validity window
5. Stitch windows into one continuous time series per satellite
6. EDA: altitude profile, TLE age distribution, coverage gaps
7. Save CSVs to `data/collected/` (or download as ZIP on Colab)

## 0 — Environment Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running on Google Colab')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'sgp4', 'python-dotenv', 'requests', 'plotly'], check=True)
else:
    print('Running locally')

import os, math, time, csv
from pathlib import Path
from datetime import datetime, timezone, timedelta

import numpy as np
import requests
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from sgp4.api import Satrec, jday

print('All imports OK')

## 1 — Credentials

| Runtime | Method |
|---|---|
| **Local** | `.env` in the project root is loaded automatically |
| **Colab — Option A** | Upload your `.env` via the file picker (runs below) |
| **Colab — Option B** | Colab Secrets (🔑 sidebar): `SPACETRACK_USER`, `SPACETRACK_PASS` |

> **Never paste raw credentials into a cell** — they are saved in the `.ipynb` JSON.

In [ ]:
if IN_COLAB:
    _already_loaded = bool(os.environ.get('SPACETRACK_USER'))
    if not _already_loaded:
        try:
            from google.colab import files as _colab_files
            print('A file-picker will open — select your .env file.')
            _uploaded = _colab_files.upload()
            if _uploaded:
                _env_bytes = next(iter(_uploaded.values()))
                for _line in _env_bytes.decode().splitlines():
                    _line = _line.strip()
                    if not _line or _line.startswith('#') or '=' not in _line:
                        continue
                    _k, _v = _line.split('=', 1)
                    os.environ[_k.strip()] = _v.strip().strip('"\'')
                print('  .env loaded into environment.')
        except Exception as _e:
            print(f'  File upload skipped: {_e}')
    else:
        print('  Credentials already in environment.')

    # Option B — Colab Secrets (uncomment if preferred):
    # from google.colab import userdata
    # os.environ.setdefault('SPACETRACK_USER', userdata.get('SPACETRACK_USER'))
    # os.environ.setdefault('SPACETRACK_PASS', userdata.get('SPACETRACK_PASS'))
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=Path('../.env'), override=False)
    except ImportError:
        pass

ST_USER = os.environ.get('SPACETRACK_USER', '')
ST_PASS = os.environ.get('SPACETRACK_PASS', '')
if not ST_USER or not ST_PASS:
    raise ValueError('Space-Track credentials not found. See instructions above.')
print(f'Credentials loaded for: {ST_USER}')

## 2 — Configuration

In [ ]:
# Satellites — LEO only, diverse altitudes and inclinations
NORAD_IDS = [
    43017,   # AO-91  (Fox-1D)     ~500 km, 65 deg inc, amateur CubeSat
    43137,   # AO-95  (Fox-1Cliff) ~500 km, 85 deg inc
    25544,   # ISS                 ~400 km, 51.6 deg inc, large / high drag
    33591,   # NOAA-19             ~870 km, 99 deg inc, sun-synchronous
    20580,   # Hubble              ~540 km, 28.5 deg inc
]

HISTORY_DAYS   = 180   # days of TLE history to fetch per satellite
PROPAGATE_STEP = 1     # minutes between position samples
STALE_DAYS     = 3     # TLE age threshold used to build SGP4-stale baseline

OUTPUT_DIR = Path('/content/collected') if IN_COLAB else Path('../data/collected')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://www.space-track.org'
print(f'Output directory : {OUTPUT_DIR.resolve()}')
print(f'Stale TLE age    : {STALE_DAYS} days')

## 3 — Fetch Historical TLEs from Space-Track

In [ ]:
def login_spacetrack(session):
    resp = session.post(f'{BASE_URL}/ajaxauth/login',
                        data={'identity': ST_USER, 'password': ST_PASS})
    resp.raise_for_status()
    if 'Failed' in resp.text:
        raise RuntimeError('Space-Track login failed — check credentials.')
    print('  Logged in to Space-Track.')

def fetch_tle_history(session, norad_id, days):
    """Return list of (line1, line2) TLE pairs for the past `days` days."""
    since = (datetime.now(timezone.utc) - timedelta(days=days)).strftime('%Y-%m-%d')
    url = (f'{BASE_URL}/basicspacedata/query/class/gp_history'
           f'/NORAD_CAT_ID/{norad_id}/orderby/EPOCH asc'
           f'/EPOCH/{since}--now/format/tle')
    resp = session.get(url)
    resp.raise_for_status()
    lines = [l.strip() for l in resp.text.strip().splitlines() if l.strip()]
    pairs = [(lines[i], lines[i+1]) for i in range(0, len(lines) - 1, 2)
             if lines[i].startswith('1') and lines[i+1].startswith('2')]
    print(f'  NORAD {norad_id}: {len(pairs)} TLEs fetched.')
    return pairs

session = requests.Session()
login_spacetrack(session)

all_tle_history = {}
for nid in NORAD_IDS:
    time.sleep(1)   # respect Space-Track rate limit (20 requests/min)
    all_tle_history[nid] = fetch_tle_history(session, nid, HISTORY_DAYS)

session.close()
print('Done fetching.')

## 4 — Propagate TLEs to Position Time Series

In [ ]:
from sgp4.api import Satrec, jday
from datetime import timedelta

def tle_epoch_dt(line1):
    epoch_str = line1[18:32].strip()
    year_2d = int(epoch_str[:2])
    year = 2000 + year_2d if year_2d < 57 else 1900 + year_2d
    day_of_year = float(epoch_str[2:])
    base = datetime(year, 1, 1, tzinfo=timezone.utc)
    return base + timedelta(days=day_of_year - 1)

def ecef_to_geodetic(x_km, y_km, z_km):
    a, b = 6378.137, 6356.752
    e2 = 1 - (b / a) ** 2
    lon = math.degrees(math.atan2(y_km, x_km))
    p = math.sqrt(x_km**2 + y_km**2)
    lat = math.degrees(math.atan2(z_km, p * (1 - e2)))
    for _ in range(5):
        sin_lat = math.sin(math.radians(lat))
        N = a / math.sqrt(1 - e2 * sin_lat**2)
        lat = math.degrees(math.atan2(z_km + e2 * N * sin_lat, p))
    sin_lat = math.sin(math.radians(lat))
    N = a / math.sqrt(1 - e2 * sin_lat**2)
    alt = (p / math.cos(math.radians(lat)) - N) if abs(lat) < 89 else (z_km / math.sin(math.radians(lat)) - N * (1 - e2))
    return lat, lon, alt

def propagate_point(satrec, t_dt):
    """Propagate one satrec to a single datetime. Returns (lat, lon, alt) or None on error."""
    jd, fr = jday(t_dt.year, t_dt.month, t_dt.day, t_dt.hour, t_dt.minute, t_dt.second)
    e, r, _ = satrec.sgp4(jd, fr)
    if e != 0:
        return None
    return ecef_to_geodetic(*r)

def find_stale_sat(satrec_list, t_dt, stale_days):
    """Return (epoch, satrec) of the most recent TLE that is >= stale_days old at t_dt."""
    cutoff = t_dt - timedelta(days=stale_days)
    result = None
    for ep, sat in satrec_list:
        if ep <= cutoff:
            result = (ep, sat)
        else:
            break   # list is sorted ascending
    return result

for norad_id, pairs in all_tle_history.items():
    print(f'Propagating NORAD {norad_id}...')

    # Build sorted (epoch, satrec) list for stale-TLE lookup
    satrec_list = []
    for l1, l2 in pairs:
        ep = tle_epoch_dt(l1)
        satrec_list.append((ep, Satrec.twoline2rv(l1, l2)))
    satrec_list.sort(key=lambda x: x[0])

    all_records, seen_times = [], set()
    for i, (l1, l2) in enumerate(pairs):
        epoch     = tle_epoch_dt(l1)
        fresh_sat = Satrec.twoline2rv(l1, l2)
        next_epoch = tle_epoch_dt(pairs[i+1][0]) if i + 1 < len(pairs) else epoch + timedelta(days=3)
        window_end = min(next_epoch, epoch + timedelta(days=3))

        t = epoch
        while t <= window_end:
            ts = t.isoformat()
            if ts in seen_times:
                t += timedelta(minutes=PROPAGATE_STEP); continue

            jd, fr = jday(t.year, t.month, t.day, t.hour, t.minute, t.second)
            e, r, v = fresh_sat.sgp4(jd, fr)
            if e == 0:
                lat, lon, alt = ecef_to_geodetic(*r)
                tle_age = (t - epoch).total_seconds() / 86400

                # --- Stale SGP4 baseline ---
                stale = find_stale_sat(satrec_list, t, STALE_DAYS)
                if stale:
                    stale_ep, stale_sat = stale
                    pos_stale = propagate_point(stale_sat, t)
                    if pos_stale:
                        s_lat, s_lon, s_alt = pos_stale
                        s_age = (t - stale_ep).total_seconds() / 86400
                    else:
                        s_lat = s_lon = s_alt = s_age = None
                else:
                    s_lat = s_lon = s_alt = s_age = None  # not enough history yet

                seen_times.add(ts)
                all_records.append({
                    'timestamp_utc'    : ts,
                    'lat_deg'          : round(lat, 6),
                    'lon_deg'          : round(lon, 6),
                    'alt_km'           : round(alt, 4),
                    'vx_km_s'          : round(v[0], 6),
                    'vy_km_s'          : round(v[1], 6),
                    'vz_km_s'          : round(v[2], 6),
                    'tle_age_days'     : round(tle_age, 4),
                    'sgp4_stale_lat'   : round(s_lat, 6)  if s_lat  is not None else None,
                    'sgp4_stale_lon'   : round(s_lon, 6)  if s_lon  is not None else None,
                    'sgp4_stale_alt'   : round(s_alt, 4)  if s_alt  is not None else None,
                    'sgp4_stale_age'   : round(s_age, 4)  if s_age  is not None else None,
                })
            t += timedelta(minutes=PROPAGATE_STEP)

    all_records.sort(key=lambda r: r['timestamp_utc'])
    out_path = OUTPUT_DIR / f'positions_{norad_id}.csv'
    if all_records:
        with open(out_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=all_records[0].keys())
            writer.writeheader()
            writer.writerows(all_records)
    n_with_stale = sum(1 for r in all_records if r['sgp4_stale_lat'] is not None)
    print(f'  Saved {len(all_records):,} records ({n_with_stale:,} with stale baseline) -> {out_path.name}')

print('All done.')

## 5 — Exploratory Data Analysis

In [ ]:
sample_norad = NORAD_IDS[0]
df = pd.read_csv(OUTPUT_DIR / f'positions_{sample_norad}.csv', parse_dates=['timestamp_utc'])
print(f'NORAD {sample_norad}: {len(df):,} records')
print(df.describe().round(3))

In [ ]:
# Altitude over time (sample 5000 points for speed)
sample = df.sample(min(5000, len(df))).sort_values('timestamp_utc')
fig = px.line(sample, x='timestamp_utc', y='alt_km',
              title=f'NORAD {sample_norad} — Altitude over Time',
              labels={'alt_km': 'Altitude (km)', 'timestamp_utc': 'UTC'})
fig.show()

# TLE age distribution
fig2 = px.histogram(df, x='tle_age_days', nbins=60,
                    title=f'NORAD {sample_norad} — TLE Age Distribution',
                    labels={'tle_age_days': 'TLE Age at Sample (days)'})
fig2.show()

# Coverage gaps
df_sorted = df.sort_values('timestamp_utc').reset_index(drop=True)
gaps_min = df_sorted['timestamp_utc'].diff().dt.total_seconds().div(60)
large_gaps = gaps_min[gaps_min > 5].dropna()
print(f'Total records: {len(df):,}')
print(f'Gaps > 5 min : {len(large_gaps)}')
print(f'Max gap      : {large_gaps.max():.1f} min' if len(large_gaps) else 'No large gaps')

## 6 — Download Data (Colab only)

Run this cell on Colab to download all position CSVs as a ZIP. You will upload this ZIP in Notebook 02.

In [ ]:
if IN_COLAB:
    import zipfile, io
    from google.colab import files as _colab_files
    zip_buf = io.BytesIO()
    with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for csv_path in OUTPUT_DIR.glob('positions_*.csv'):
            zf.write(csv_path, csv_path.name)
    zip_buf.seek(0)
    with open('/content/collected_positions.zip', 'wb') as f:
        f.write(zip_buf.read())
    _colab_files.download('/content/collected_positions.zip')
    print('Download started — save this ZIP, you will upload it in Notebook 02.')
else:
    print(f'Local: files are in {OUTPUT_DIR.resolve()}')